In [ ]:
import os
import time
import random
import numpy as np 
import pandas as pd

import cv2
from PIL import Image
import albumentations as A

import torch
import torch.nn as nn
from tqdm.notebook import tqdm
import torch.nn.functional as F
from torchvision import transforms as T
import segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)
set_seed(0)

In [ ]:
def create_df(IMAGE_PATH):
    name = []
    for dirname, _, filenames in os.walk(IMAGE_PATH):
        for filename in filenames:
            name.append(filename.split('.')[0])
    return pd.DataFrame({'id': name}, index = np.arange(0, len(name)))

In [ ]:
def rgb_to_class_mask(rgb_mask, color_to_class):
    class_mask = np.zeros((rgb_mask.shape[0], rgb_mask.shape[1]), dtype=np.int64)
    for color, class_index in color_to_class.items():
        class_mask[np.all(rgb_mask == color, axis=-1)] = class_index
    return class_mask

In [ ]:
class CloudDataset(Dataset):
    def __init__(self, img_path, mask_path, X, mean, std, transform=None, patch=False):
        self.img_path = img_path
        self.mask_path = mask_path
        self.X = X
        self.transform = transform
        self.patches = patch
        self.mean = mean
        self.std = std
        self.color_to_class = {
            (  0,   0,   0): 0, # Clear
            (255,  75,  75): 1, # Cirrus
            (255, 150,   0): 2, # Cirrostratus
            (255, 210,  75): 3, # Stratus
            (175, 225, 130): 4, # Stratocumulus
            (130, 175, 225): 5, # Cumulus
            (175, 150, 255): 6, # Cirrocumulus
            (150, 150, 150): 7, # Nimbus
        }

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        img = cv2.imread(self.img_path + self.X[idx] + '.jpg')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = np.array(cv2.imread(self.mask_path + self.X[idx] + '_mask.png', cv2.IMREAD_COLOR))
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        mask = rgb_to_class_mask(mask, self.color_to_class)
        
        if img.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)

        if self.transform is not None:
            aug = self.transform(image=img, mask=mask)
            img = Image.fromarray(aug['image'])
            mask = aug['mask']
        
        if self.transform is None:
            img = Image.fromarray(img)
        
        t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        img = t(img)
        mask = torch.from_numpy(mask).long()
        
        if self.patches:
            img, mask = self.tiles(img, mask)
            
        return img, mask
    
    def tiles(self, img, mask):
        # tensor.contiguous().view() is similar in functionality to numpy.reshape
        img_patches = img.unfold(1, 512, 512).unfold(2, 768, 768) 
        img_patches = img_patches.contiguous().view(3, -1, 512, 768)
        img_patches = img_patches.permute(1,0,2,3)

        mask_patches = mask.unfold(1, 512, 512).unfold(2, 768, 768)
        mask_patches = mask_patches.contiguous().view(3, -1, 512, 768)
        mask_patches = mask_patches.permute(1,0,2,3)
        
        return img_patches, mask_patches

In [ ]:
batch_size = 4
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

In [ ]:
# Data Augumentation
t_train = A.Compose([A.Resize(320, 416, interpolation=cv2.INTER_AREA), A.HorizontalFlip()])
t_val = A.Compose([A.Resize(320, 416, interpolation=cv2.INTER_AREA)])

In [ ]:
model = smp.Unet('timm-mobilenetv3_large_100', encoder_weights='imagenet', classes=8,
                 activation=None, encoder_depth=5, decoder_channels=[256, 128, 64, 32, 16])

In [ ]:
def pixel_accuracy(output, mask):
    with torch.no_grad():
        output = torch.argmax(F.softmax(output, dim=1), dim=1)
        correct = torch.eq(output, mask).int()
        accuracy = float(correct.sum()) / float(correct.numel())
    return accuracy

In [ ]:
def wiou(pred_mask, mask, smooth=1e-10, n_classes=8):
    with torch.no_grad():
        pred_mask = F.softmax(pred_mask, dim=1)
        pred_mask = torch.argmax(pred_mask, dim=1)

        if mask.dim() == 4:
            mask = mask.squeeze(1)

        batch_size = pred_mask.size(0)
        batch_ious = []
        for b in range(batch_size):
            pred_flat = pred_mask[b].contiguous().view(-1)
            mask_flat = mask[b].contiguous().view(-1)
            iou_per_class = []
            total_true_label_count = 0
            for clas in range(n_classes):
                true_class = pred_flat == clas
                true_label = mask_flat == clas
                true_label_count = true_label.long().sum().item()
                total_true_label_count += true_label_count

                if true_label_count == 0: # no exist label in this loop
                    iou_per_class.append(np.nan)
                else:
                    intersect = torch.logical_and(true_class, true_label).sum().float().item()
                    union = torch.logical_or(true_class, true_label).sum().float().item()
                    # add a smoothing term to avoid division by zero
                    iou = (intersect + smooth) / (union + smooth)
                    weight_iou = true_label_count * iou
                    iou_per_class.append(weight_iou)

            valid_iou = [iou for iou in iou_per_class if not np.isnan(iou)]
            img_iou = np.sum(valid_iou) / total_true_label_count if total_true_label_count > 0 else 0.0
            batch_ious.append(img_iou)
        return np.mean(batch_ious)

In [ ]:
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group['lr']

def fit(epochs, model, train_loader, val_loader, criterion, optimizer, scheduler, patch=False):
    torch.cuda.empty_cache()
    train_losses, val_losses = [], []
    train_iou, train_acc, val_iou, val_acc = [], [], [], []
    lrs = []
    min_loss = np.inf
    max_acc = -np.inf
    max_wiou = -np.inf

    model.to(device)
    fit_time = time.time()
    for e in range(epochs):
        since = time.time()

        running_loss = 0
        iou_score = 0
        accuracy = 0

        # training loop
        model.train()
        for i, data in enumerate(tqdm(train_loader)):
            # training phase
            image_tiles, mask_tiles = data
            if patch:
                bs, n_tiles, c, h, w = image_tiles.size()
                image_tiles = image_tiles.view(-1, c, h, w)
                mask_tiles = mask_tiles.view(-1, c, h, w)
            image, mask = image_tiles.to(device), mask_tiles.to(device)

            # forward
            output = model(image)
            loss = criterion(output, mask)
            # evaluation metrics
            iou_score += wiou(output, mask)
            accuracy += pixel_accuracy(output, mask)

            # backward
            loss.backward()
            optimizer.step() # update weight          
            optimizer.zero_grad() # reset gradient
            
            # step the learning rate
            lrs.append(get_lr(optimizer))
            scheduler.step() 
            
            running_loss += loss.item()
            
        else:
            model.eval()
            val_loss = 0
            val_accuracy = 0
            val_iou_score = 0
            # validation loop
            with torch.no_grad():
                for i, data in enumerate(tqdm(val_loader)):
                    # validation phase
                    image_tiles, mask_tiles = data
                    if patch:
                        bs, n_tiles, c, h, w = image_tiles.size()
                        image_tiles = image_tiles.view(-1, c, h, w)
                        mask_tiles = mask_tiles.view(-1, c, h, w)
                    image, mask = image_tiles.to(device), mask_tiles.to(device)

                    output = model(image)

                    val_iou_score += wiou(output, mask)
                    val_accuracy += pixel_accuracy(output, mask)

                    loss = criterion(output, mask)                                  
                    val_loss += loss.item()
            
            # calculate mean for each batch
            train_losses.append(running_loss/len(train_loader))
            val_losses.append(val_loss/len(val_loader))

            if min_loss > (val_loss/len(val_loader)):
                print('Loss Decreasing.. {:.3f} >> {:.3f}'.format(min_loss, (val_loss/len(val_loader))))
                min_loss = (val_loss/len(val_loader))
                print('saving model...')
                torch.save(model, f'./Output/Unet-wiou-{val_iou_score/len(val_loader):.3f}_loss-{min_loss:.3f}.pt')
            
            elif max_acc < (val_accuracy/len(val_loader)):
                print('Acc Increasing.. {:.3f} >> {:.3f}'.format(max_acc, (val_accuracy/len(val_loader))))
                max_acc = (val_accuracy/len(val_loader))
                print('saving model...')
                torch.save(model, f'./Output/Unet-wiou-{val_iou_score/len(val_loader):.3f}_acc-{max_acc:.3f}.pt')

            elif max_wiou < (val_iou_score/len(val_loader)):
                print('wiou Increasing.. {:.3f} >> {:.3f}'.format(max_wiou, (val_iou_score/len(val_loader))))
                max_wiou = val_iou_score/len(val_loader)
                print('saving model...')
                torch.save(model, f'./Output/Unet-wiou-{val_iou_score/len(val_loader):.3f}.pt')

            # iou
            train_iou.append(iou_score/len(train_loader))
            val_iou.append(val_iou_score/len(val_loader))
            train_acc.append(accuracy/len(train_loader))
            val_acc.append(val_accuracy/ len(val_loader))
            print("Epoch:{}/{}..".format(e+1, epochs),
                  "Train Loss:{:.3f}..".format(running_loss/len(train_loader)),
                  "Val Loss:{:.3f}..".format(val_loss/len(val_loader)),
                  "Train wiou:{:.3f}..".format(iou_score/len(train_loader)),
                  "Val wiou:{:.3f}..".format(val_iou_score/len(val_loader)),
                  "Train Acc:{:.3f}..".format(accuracy/len(train_loader)),
                  "Val Acc:{:.3f}..".format(val_accuracy/len(val_loader)),
                  "Time:{:.2f} m".format((time.time()-since)/60))
        
    history = {'train_loss' : train_losses, 'val_loss' : val_losses,
               'train_wiou' : train_iou, 'val_wiou' : val_iou,
               'train_acc' : train_acc, 'val_acc' : val_acc, 'lrs': lrs} 
    print('Total time: {:.2f} m' .format((time.time()-fit_time)/60))
    return history

In [ ]:
output_dir = './Output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)

In [ ]:
IMAGE_PATH_TRAIN = './Data/train/images/'
MASK_PATH_TRAIN = './Data/train/masks/'

IMAGE_PATH_VAL = './Data/valid/images/'
MASK_PATH_VAL = './Data/valid/masks/'

df_train = create_df(IMAGE_PATH_TRAIN)
df_val = create_df(IMAGE_PATH_VAL)

X_train = df_train['id'].values
X_val = df_val['id'].values

train_set = CloudDataset(IMAGE_PATH_TRAIN, MASK_PATH_TRAIN, X_train, mean, std, t_train, patch=False)
val_set = CloudDataset(IMAGE_PATH_VAL, MASK_PATH_VAL, X_val, mean, std, t_val, patch=False)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)


max_lr = 1e-3
epoch = 100
weight_decay = 1e-4

# Assigning higher weights to underrepresented classes (Cirrus and Cumulus)
class_weights = torch.tensor([1.0, 5.0, 1.0, 1.0, 1.0, 3.0, 1.0, 1.0]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epoch*len(train_loader), eta_min=1e-6)

history = fit(epoch, model, train_loader, val_loader, criterion, optimizer, sched)

history_copy = history.copy()
history_copy.pop('lrs')
history_df = pd.DataFrame(history_copy)
history_df.to_csv(f"training_history_target.csv", index_label='epoch')

In [ ]:
# Saving the best model's state_dict for future loading
# import torch
# model = torch.load('./Unet-wiou-0.721_loss-0.777.pt', weights_only=False)
# torch.save(model.state_dict(), './Unet-wiou-0.721_loss-0.777.pt')